# Automatic Music Transcription

In [1]:
# install relevant packages for Google Colab
# !pip install pyguitarpro
# !pip install --upgrade attrs
# !pip install accelerate -U
# !pip install datasets

### Preprocessing

In [2]:
from sklearn.model_selection import train_test_split
import IPython.display as ipd
import copy
import torch
import torch.nn.functional as F
import librosa
import guitarpro
from guitarpro.models import Note, Beat
from transformers import AutoFeatureExtractor, WhisperModel, WhisperConfig, WhisperForConditionalGeneration, WhisperProcessor


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'{device}, {torch.__version__=}, {torch.cuda.is_available()=}, {torch.version.cuda=}')

cuda, torch.__version__='2.1.2', torch.cuda.is_available()=True, torch.version.cuda='12.1'


In [3]:
song_filepath = 'dataset/john-mayer/30_Why_Georgia.mp3'
tab_filepath = 'dataset/john-mayer/3_why-georgia.gp5'
vocab_size = 129
wait_token_idx = 127
start_token_idx = 128
end_token_idx = 129

sr = 16000

model_id = "openai/whisper-base"

# Load in the song and tab
y, sr = librosa.load(song_filepath, sr=sr)
song = guitarpro.parse(tab_filepath) 

print(f'{len(y)=}')

len(y)=484922


In [4]:
def tokenize_note_simple(note):
    """
    Tokenizes a note based on its string and fret number.
    """

    string = note.string
    value = note.value

    # Add pre-conditions for the note
    if string < 1 or string > 6:
        raise ValueError(f"Invalid string number: {string}")

    if value < 0 or value > 20:
        raise ValueError(f"Invalid fret number: {value}")

    # Calculate the token index based on string and fret
    # Token range for fretted notes: 1 to 120
    # Token range for open strings: 121 to 126
    if value == 0:  # Open string
        token_index = 121 + (6 - string)
    else:
        token_index = (string - 1) * 20 + value

    return token_index

In [5]:
g_note = Note(Beat(voice=None), string=6, value=3)
g_note_token_idx = tokenize_note_simple(g_note)

print(f'{g_note_token_idx=} for {g_note}')

g_note_token_idx=103 for Note(value=3, velocity=95, string=6, effect=<guitarpro.models.NoteEffect object at 0x7f9ba3b300d0>, durationPercent=1.0, swapAccidentals=False, type=<NoteType.rest: 0>)


In [6]:
def detokenize_note_simple(token_index):
    """
    Detokenizes a token index to find the corresponding guitar string and fret number
    """

    # Pre-conditions for the token index
    if token_index < 1 or token_index > vocab_size:
        raise ValueError(f"Invalid token index: {token_index}")

    # Determining the string and fret based on token index
    if 121 <= token_index <= 126:  # Open string
        string = 6 - (token_index - 121)
        fret = 0
    elif token_index == wait_token_idx:
        return "wait"
    elif token_index == start_token_idx:
        return "start"
    elif token_index == end_token_idx:
        return "end"
    else:
        string = ((token_index - 1) // 20) + 1
        fret = token_index - ((string - 1) * 20)

    return f"s{string}f{fret}"

# Example usage
token_index = 126
note = detokenize_note_simple(token_index)
print(f'{token_index=} maps to {note=}')


token_index=126 maps to note='s1f0'


In [7]:
def tokenize_song_simple(song):
    """
    Tokenizes an entire song using a simplified scheme.
    """
    all_measures_tokens = []

    # Start token for the song
    all_measures_tokens.append([128])  # 128 is the start token

    for track in song.tracks:
        for measure in track.measures:
            measure_tokens = []
            for voice in measure.voices:
                for beat in voice.beats:
                    if track.isPercussionTrack:
                        continue  # Skip percussion tracks
                    else:
                        for note in beat.notes:
                            token = tokenize_note_simple(note)
                            measure_tokens.append(token)

                    # Assuming each beat is a 'wait' token
                    measure_tokens.append(127)  # 127 is the wait/rest token

            # Repeat measures if necessary
            if measure.repeatClose > 0:
                for _ in range(measure.repeatClose):
                    all_measures_tokens.append(measure_tokens)
            else:
                all_measures_tokens.append(measure_tokens)

    # End token for the song
    all_measures_tokens.append([129])  # 129 is the end token

    return all_measures_tokens

In [8]:
# tokenize the acoustic song track only
acoustic_song = copy.deepcopy(song)
acoustic_song.tracks = [acoustic_song.tracks[0]]

tokens = tokenize_song_simple(acoustic_song)

## Audio Segmentation
* Split the audio into segments of roughly the same length as the song measures, which for Why Georgia is roughly 2.5 seconds

In [9]:
# Given parameters
sample_rate = sr
chunk_duration = 2.5  # Chunk duration in seconds, this is the approximate length for each measure
hop_length = 512  # This is the default hop length for librosa's mfcc calculation

# Calculate the number of samples per chunk
samples_per_chunk = int(sample_rate * chunk_duration)

# Calculate the number of chunks
num_chunks = len(y) // samples_per_chunk + 1

# calculate total number of frames
total_frames = len(y) // hop_length + 1

# Calculate the number of frames per chunk
# Formula: Number of Frames = (Number of Samples in Segment - Frame Length) / Hop Length + 1
# Assuming the frame length is equal to the hop length for simplicity
frames_per_chunk = (samples_per_chunk) // hop_length + 1

print(f'{samples_per_chunk=}, {num_chunks=}, {frames_per_chunk=}, {total_frames=}')

samples_per_chunk=40000, num_chunks=13, frames_per_chunk=79, total_frames=948


In [10]:
def extract_audio_segments(y, num_chunks, samples_per_chunk):
    """
    Extracts audio segments from the given waveform y into num_chunks segments.
    """

    # Initialize an empty list to hold the audio segments
    audio_segments = []

    # Iterate over the waveform and extract segments
    for i in range(num_chunks):
        # Calculate the start and end index for each chunk
        start_idx = i * samples_per_chunk
        end_idx = start_idx + samples_per_chunk

        # Make sure the end index does not exceed the length of the waveform
        end_idx = min(end_idx, len(y))

        # Extract the segment and add it to the list
        segment = y[start_idx:end_idx]
        audio_segments.append(segment)

    audio_segments = audio_segments[:-1]  # remove the last segment as it may not be a full measure
    segment_len = len(audio_segments[0])
    
    for segment in audio_segments:
        assert len(segment) == segment_len, "All segments should have the same length"
    
    return audio_segments

audio_segments = extract_audio_segments(y, num_chunks, samples_per_chunk)

### Use a seq2seq transformer model

In [11]:
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
processor = WhisperProcessor.from_pretrained(model_id)

In [12]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"


In [13]:
class CustomWhisperModel(WhisperForConditionalGeneration):
    def __init__(self, config: WhisperConfig, vocab_size: int):
        super().__init__(config)
        # Initialize your custom projection layer
        # self.proj_out = torch.nn.Linear(384, vocab_size, bias=False)

    def forward(self, **kwargs):
        outputs = super().forward(**kwargs)
        
        return outputs

# Load the original model configuration
config = WhisperConfig()
vocab_size = 129  # Your vocab size
config.vocab_size = vocab_size
config.pad_token_id = wait_token_idx
config.decoder_start_token_id = start_token_idx
config.eos_token_id = end_token_idx
config.bos_token_id = start_token_idx
config.begin_suppress_tokens = None

# Initialize your custom model with the original configuration and your vocab_size
custom_model = CustomWhisperModel(config, vocab_size)

# Move the model to the device
custom_model.to(device)

CustomWhisperModel(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 384, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(384, 384, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 384)
      (layers): ModuleList(
        (0-3): 4 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=384, out_features=384, bias=False)
            (v_proj): Linear(in_features=384, out_features=384, bias=True)
            (q_proj): Linear(in_features=384, out_features=384, bias=True)
            (out_proj): Linear(in_features=384, out_features=384, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (final_layer_

In [21]:
# lets loop through each audio segment and generate the tab
for segment in audio_segments[:1]:
    input_features = processor(segment, sampling_rate=sr, return_tensors="pt").input_features.to(device)
    # custom_model.config.forced_decoder_ids = None
    print(f'{torch.isnan(input_features).any().item(), torch.isinf(input_features).any().item()}')
    logits = custom_model.generate(input_features, max_length=100)

    print(f'{logits.shape}')
    print(f'{logits=}\n')
    # predicted_token_id = torch.argmax(logits, dim=-1).item()
    # predicted_note = detokenize_note_simple(predicted_token_id)
    # print(predicted_note, predicted_token_id)

(False, False)
torch.Size([1, 100])
logits=tensor([[128, 110, 110, 102, 102,  88,  88,  88,  88,  88,  88,  88,  88,  88,
          88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,
          88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,
          88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,
          88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,
          88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,
          88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,  88,
          88,  88]], device='cuda:0')



In [15]:
custom_model.config

WhisperConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "apply_spec_augment": false,
  "attention_dropout": 0.0,
  "begin_suppress_tokens": null,
  "bos_token_id": 128,
  "classifier_proj_size": 256,
  "d_model": 384,
  "decoder_attention_heads": 6,
  "decoder_ffn_dim": 1536,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 4,
  "decoder_start_token_id": 128,
  "dropout": 0.0,
  "encoder_attention_heads": 6,
  "encoder_ffn_dim": 1536,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 4,
  "eos_token_id": 129,
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "mask_feature_length": 10,
  "mask_feature_min_masks": 0,
  "mask_feature_prob": 0.0,
  "mask_time_length": 10,
  "mask_time_min_masks": 2,
  "mask_time_prob": 0.05,
  "max_source_positions": 1500,
  "max_target_positions": 448,
  "median_filter_width": 7,
  "model_type": "whisper",
  "num_hidden_layers": 4,
  "num_mel_bins": 80,
  "pad_token_id": 127,
  "scale_embedding": false,
  "transformers_version": "4.